# **Transformer Model Outline**

A basic transformer based neural network trained on Haruki Murakami's After Dark.

Prerequisite knowlege required. See [Spoticore](../../../spoticore/src/spoticore/).


In [2]:
BOOK_PATH = "./book.txt"

with open(BOOK_PATH, "r", encoding="utf-8") as bk:
    text = bk.read()
    lines = text.splitlines()

print("total characters in book:\n", len(text))
print("\nfirst 1000 characters:\n", text[:1000])
print("\nfirst 5 lines:\n", lines[:5])

total characters in book:
 258438

first 1000 characters:
 Eyes mark the shape of the city.

Through the eyes of a high-flying night bird, we take in the scene from midair. In our broad sweep, the city looks like a single gigantic creature—or more like a single collective entity created by many intertwining organisms. Countless arteries stretch to the ends of its elusive body, circulating a continuous supply of fresh blood cells, sending out new data and collecting the old, sending out new consumables and collecting the old, sending out new contradictions and collecting the old. To the rhythm of its pulsing, all parts of the body flicker and flare up and squirm. Midnight is approaching, and while the peak of activity has passed, the basal metabolism that maintains life continues undiminished, producing the basso continuo of the city’s moan, a monotonous sound that neither rises nor falls but is pregnant with foreboding.

Our line of sight chooses an area of concentrated brightness and,

## Text preprocessing and analysis


#### Unique characters in dataset


In [3]:
chars = sorted(set(text))
print("unique characters:", chars)

chars_str = "".join(chars)
print("\nunique characters as string:", chars_str)

vocab_size = len(chars)
print("\nvocabulary size:", vocab_size)

unique characters: ['\n', ' ', '!', '(', ')', ',', '-', '.', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', ':', ';', '?', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'R', 'S', 'T', 'U', 'V', 'W', 'Y', 'Z', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z', 'à', 'â', 'è', 'é', '—', '‘', '’', '“', '”', '…']

unique characters as string: 
 !(),-.0123456789:;?ABCDEFGHIJKLMNOPRSTUVWYZabcdefghijklmnopqrstuvwxyzàâèé—‘’“”…

vocabulary size: 81


#### StoI and ItoS mappings


In [4]:
stoi = {c: i for i, c in enumerate(chars)}
itos = {i: c for c, i in stoi.items()}

print(f"{stoi = }\n\n{itos = }")

stoi = {'\n': 0, ' ': 1, '!': 2, '(': 3, ')': 4, ',': 5, '-': 6, '.': 7, '0': 8, '1': 9, '2': 10, '3': 11, '4': 12, '5': 13, '6': 14, '7': 15, '8': 16, '9': 17, ':': 18, ';': 19, '?': 20, 'A': 21, 'B': 22, 'C': 23, 'D': 24, 'E': 25, 'F': 26, 'G': 27, 'H': 28, 'I': 29, 'J': 30, 'K': 31, 'L': 32, 'M': 33, 'N': 34, 'O': 35, 'P': 36, 'R': 37, 'S': 38, 'T': 39, 'U': 40, 'V': 41, 'W': 42, 'Y': 43, 'Z': 44, 'a': 45, 'b': 46, 'c': 47, 'd': 48, 'e': 49, 'f': 50, 'g': 51, 'h': 52, 'i': 53, 'j': 54, 'k': 55, 'l': 56, 'm': 57, 'n': 58, 'o': 59, 'p': 60, 'q': 61, 'r': 62, 's': 63, 't': 64, 'u': 65, 'v': 66, 'w': 67, 'x': 68, 'y': 69, 'z': 70, 'à': 71, 'â': 72, 'è': 73, 'é': 74, '—': 75, '‘': 76, '’': 77, '“': 78, '”': 79, '…': 80}

itos = {0: '\n', 1: ' ', 2: '!', 3: '(', 4: ')', 5: ',', 6: '-', 7: '.', 8: '0', 9: '1', 10: '2', 11: '3', 12: '4', 13: '5', 14: '6', 15: '7', 16: '8', 17: '9', 18: ':', 19: ';', 20: '?', 21: 'A', 22: 'B', 23: 'C', 24: 'D', 25: 'E', 26: 'F', 27: 'G', 28: 'H', 29: 'I', 30

#### encoder and decoder


In [5]:
def encode(s: str) -> list[int]:
    return [stoi[c] for c in s]


def decode(encoding: list[int]) -> str:
    return "".join([itos[encode] for encode in encoding])


ex = "hi there i am vihanga"
print("example:", ex)
enc = encode(ex)
print("\nencoding:", enc)
dec = decode(enc)
print("\ndecoding:", dec)

example: hi there i am vihanga

encoding: [52, 53, 1, 64, 52, 49, 62, 49, 1, 53, 1, 45, 57, 1, 66, 53, 52, 45, 58, 51, 45]

decoding: hi there i am vihanga


### Encode book into tensor


In [6]:
from typing import Final
import torch

SEED: Final[int] = 4324

torch.manual_seed(SEED)


data = torch.tensor(encode(text), dtype=torch.long)
print(f"{data.shape = } {data.dtype = }")
print(f"\nfirst 1000 characters in gpt's pov:\n{data[:1000]}")

data.shape = torch.Size([258438]) data.dtype = torch.int64

first 1000 characters in gpt's pov:
tensor([25, 69, 49, 63,  1, 57, 45, 62, 55,  1, 64, 52, 49,  1, 63, 52, 45, 60,
        49,  1, 59, 50,  1, 64, 52, 49,  1, 47, 53, 64, 69,  7,  0,  0, 39, 52,
        62, 59, 65, 51, 52,  1, 64, 52, 49,  1, 49, 69, 49, 63,  1, 59, 50,  1,
        45,  1, 52, 53, 51, 52,  6, 50, 56, 69, 53, 58, 51,  1, 58, 53, 51, 52,
        64,  1, 46, 53, 62, 48,  5,  1, 67, 49,  1, 64, 45, 55, 49,  1, 53, 58,
         1, 64, 52, 49,  1, 63, 47, 49, 58, 49,  1, 50, 62, 59, 57,  1, 57, 53,
        48, 45, 53, 62,  7,  1, 29, 58,  1, 59, 65, 62,  1, 46, 62, 59, 45, 48,
         1, 63, 67, 49, 49, 60,  5,  1, 64, 52, 49,  1, 47, 53, 64, 69,  1, 56,
        59, 59, 55, 63,  1, 56, 53, 55, 49,  1, 45,  1, 63, 53, 58, 51, 56, 49,
         1, 51, 53, 51, 45, 58, 64, 53, 47,  1, 47, 62, 49, 45, 64, 65, 62, 49,
        75, 59, 62,  1, 57, 59, 62, 49,  1, 56, 53, 55, 49,  1, 45,  1, 63, 53,
        58, 51, 56, 49, 

### Split text into train and validation datasets


In [7]:
lim = int(0.9 * len(data))
train_data = data[:lim]
val_data = data[lim:]

train_data.shape, val_data.shape

(torch.Size([232594]), torch.Size([25844]))

In [8]:
block_size = 8
print("english:", text[: block_size + 1])
print("encoding:", train_data[: block_size + 1])

english: Eyes mark
encoding: tensor([25, 69, 49, 63,  1, 57, 45, 62, 55])


#### N-1 examples in a sample chunk of size N

This will also help the model to generate predictions from smaller context sizes.


In [9]:
x = train_data[:block_size]
y = train_data[1 : block_size + 1]

print(f"{x = }]\n{y = }\n")

for i in range(block_size):
    print(f"Input: {x[: i + 1]} --> Target: {y[i]}")

x = tensor([25, 69, 49, 63,  1, 57, 45, 62])]
y = tensor([69, 49, 63,  1, 57, 45, 62, 55])

Input: tensor([25]) --> Target: 69
Input: tensor([25, 69]) --> Target: 49
Input: tensor([25, 69, 49]) --> Target: 63
Input: tensor([25, 69, 49, 63]) --> Target: 1
Input: tensor([25, 69, 49, 63,  1]) --> Target: 57
Input: tensor([25, 69, 49, 63,  1, 57]) --> Target: 45
Input: tensor([25, 69, 49, 63,  1, 57, 45]) --> Target: 62
Input: tensor([25, 69, 49, 63,  1, 57, 45, 62]) --> Target: 55


### Generate random training batches


In [10]:
from enum import Enum


class BatchType(str, Enum):
    train = "data_training_batch"
    val = "data_validation_batch"

In [11]:
# how many independent sequences processed in parallel
batch_size = 4
# max context length for a prediction
block_size = 8

# * in our case: 1 token --> 1 character


def get_batch(type: BatchType) -> torch.Tensor:
    data = train_data if type == BatchType.train else val_data
    # get batch_size random start indices
    ix = torch.randint(len(data) - block_size, (batch_size,))
    # for each batch, extract block_size consecutive elements from start
    x = torch.stack([data[i : i + block_size] for i in ix])
    # extract targets for each batch
    y = torch.stack([data[i + 1 : i + 1 + block_size] for i in ix])

    return x, y


xb, yb = get_batch(BatchType.train)

print(f"Inputs shape: {xb.shape}\nOutput shape: {yb.shape}")
print(f"\nInputs: {xb}\nOutputs: {yb}\n")

for i in range(batch_size):  # batch dimension
    for j in range(block_size):  # time dimension
        print(f"Input: {xb[i, j]} --> Target: {yb[i, j]}")

Inputs shape: torch.Size([4, 8])
Output shape: torch.Size([4, 8])

Inputs: tensor([[59, 58,  5,  1, 62, 49, 47, 49],
        [50, 50, 63,  1, 60, 45, 60, 49],
        [64, 52, 49, 57,  7, 79,  0,  0],
        [53, 58, 51,  1, 52, 49, 62, 63]])
Outputs: tensor([[58,  5,  1, 62, 49, 47, 49, 53],
        [50, 63,  1, 60, 45, 60, 49, 62],
        [52, 49, 57,  7, 79,  0,  0, 33],
        [58, 51,  1, 52, 49, 62, 63, 49]])

Input: 59 --> Target: 58
Input: 58 --> Target: 5
Input: 5 --> Target: 1
Input: 1 --> Target: 62
Input: 62 --> Target: 49
Input: 49 --> Target: 47
Input: 47 --> Target: 49
Input: 49 --> Target: 53
Input: 50 --> Target: 50
Input: 50 --> Target: 63
Input: 63 --> Target: 1
Input: 1 --> Target: 60
Input: 60 --> Target: 45
Input: 45 --> Target: 60
Input: 60 --> Target: 49
Input: 49 --> Target: 62
Input: 64 --> Target: 52
Input: 52 --> Target: 49
Input: 49 --> Target: 57
Input: 57 --> Target: 7
Input: 7 --> Target: 79
Input: 79 --> Target: 0
Input: 0 --> Target: 0
Input: 0 --> 

## Baseline: Bigram Language Model


In [12]:
import torch.nn as nn
from torch.nn import functional as F

### nn.Embedding is dimension-agnostic for input

Input shape:

$$(\text{d1, d2, d3, ..., dn})$$

Output shape:

$$(\text{d1, d2, d3, ..., dn, }{embedding\ dim})$$

Each scalar integer is replaced with its corresponding embedding vector, adding one extra dimension at the end.

Examples:


In [13]:
embedding = nn.Embedding(100, 64)

# 1D input - single sequence
idxs = torch.tensor([5, 12, 34, 8])  # shape: (4,)
out = embedding(idxs)  # shape: (4, 64)
print(f"1D output shape: {out.shape}")

# 2D input - batch of sequences
idxs = torch.randint(0, 100, (8, 20))  # shape: (8, 20)
out = embedding(idxs)  # shape: (8, 20, 64)
print(f"2D output shape: {out.shape}")

# 3D input - batch of 2D structures
idxs = torch.randint(0, 100, (4, 10, 10))  # shape: (4, 10, 10)
out = embedding(idxs)  # shape: (4, 10, 10, 64)
print(f"3D output shape: {out.shape}")

1D output shape: torch.Size([4, 64])
2D output shape: torch.Size([8, 20, 64])
3D output shape: torch.Size([4, 10, 10, 64])


In [17]:
class BigramLanguageModel(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        # bigram model is just a 2d lookup table.
        # analogus to Embedding module where lookups are logits.
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idxs: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        # idxs & targets are of shape (B, T)
        # logits are of shape (B, T, C)
        logits = self.token_embedding_table(idxs)
        B, T, C = logits.shape
        # ! cross_entropy() requires dim 1 to be channel dim
        logits, targets = logits.view(B * T, C), targets.view(B * T)
        loss = F.cross_entropy(logits, targets)

        return logits, loss


bmodel = BigramLanguageModel()
logits, loss = bmodel(xb, yb)